# SBERT + XGBoost Text Model + Late Fusion
**AI-Based Dog Health Monitoring — Team 193 · PES University**

This notebook:
1. Loads the synthetic symptom dataset (`symptom_data.csv`) for 6 dog skin disease classes
2. Optionally augments with Kaggle pet health data
3. Encodes symptoms using Sentence-BERT (`all-MiniLM-L6-v2`, 384-dim embeddings)
4. Trains an XGBoost classifier with L1/L2 regularisation and early stopping
5. Evaluates using a multi-variant protocol (5 held-out descriptions per class)
6. Performs weighted late fusion with the ResNet50 image model and finds the optimal weight via grid search

## Section 0 — Install Dependencies

In [ ]:
# Run once, then restart kernel if needed
!pip install -q sentence-transformers xgboost scikit-learn kaggle matplotlib seaborn

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

random.seed(42)
np.random.seed(42)

# ── Configuration ────────────────────────────────────────────────────────────
SYMPTOM_CSV    = './symptom_data.csv'          # synthetic dataset (relative to src/)
XGB_MODEL_PATH = './xgb_model.pkl'
LE_PATH        = './label_encoder.pkl'
IMAGE_MODEL_PATH = '../best_model.keras'       # ResNet50 (relative to src/)
IMAGE_DIR      = '../processed_image'          # image dataset root
USE_KAGGLE     = False                         # set True if kaggle API key is configured
KAGGLE_DATASET = 'yyzz1010/pet-health-symptoms'
KAGGLE_DOWNLOAD_DIR = './kaggle_data'

CLASS_NAMES = ['Dermatitis', 'Fungal_infections', 'Healthy',
               'Hypersensitivity', 'demodicosis', 'ringworm']

print('Setup complete. Classes:', CLASS_NAMES)

## Section 1 — Load Synthetic Symptom Dataset

In [ ]:
df_synthetic = pd.read_csv(SYMPTOM_CSV)
print(f'Loaded {len(df_synthetic)} synthetic entries')
print(df_synthetic['label'].value_counts())
df_synthetic.head(3)

## Section 2 — Optional Kaggle Augmentation

Set `USE_KAGGLE = True` in the config cell and ensure `~/.kaggle/kaggle.json` is in place.

In [ ]:
# Keyword map: Kaggle disease label → our class name
KAGGLE_KEYWORD_MAP = {
    'dermatitis': 'Dermatitis',
    'atopic': 'Dermatitis',
    'contact dermatitis': 'Dermatitis',
    'fungal': 'Fungal_infections',
    'malassezia': 'Fungal_infections',
    'yeast': 'Fungal_infections',
    'seborrhea': 'Fungal_infections',
    'aspergillus': 'Fungal_infections',
    'healthy': 'Healthy',
    'normal': 'Healthy',
    'no disease': 'Healthy',
    'hypersensitivity': 'Hypersensitivity',
    'allerg': 'Hypersensitivity',
    'anaphylaxis': 'Hypersensitivity',
    'demodicosis': 'demodicosis',
    'demodex': 'demodicosis',
    'mange': 'demodicosis',
    'demodectic': 'demodicosis',
    'ringworm': 'ringworm',
    'dermatophytosis': 'ringworm',
    'trichophyton': 'ringworm',
    'microsporum': 'ringworm',
}

def map_kaggle_label(raw_label):
    raw_lower = str(raw_label).lower().strip()
    for keyword, mapped in KAGGLE_KEYWORD_MAP.items():
        if keyword in raw_lower:
            return mapped
    return None

df_kaggle = pd.DataFrame()

if USE_KAGGLE:
    import kaggle
    os.makedirs(KAGGLE_DOWNLOAD_DIR, exist_ok=True)
    kaggle.api.dataset_download_files(KAGGLE_DATASET, path=KAGGLE_DOWNLOAD_DIR, unzip=True)

    # Find the first CSV in the download directory
    csv_files = [f for f in os.listdir(KAGGLE_DOWNLOAD_DIR) if f.endswith('.csv')]
    if not csv_files:
        print('No CSV found in Kaggle download. Check dataset structure.')
    else:
        raw = pd.read_csv(os.path.join(KAGGLE_DOWNLOAD_DIR, csv_files[0]))
        print('Kaggle columns:', raw.columns.tolist())

        # Identify text and label columns
        text_col  = next((c for c in raw.columns if 'symptom' in c.lower() or 'description' in c.lower() or 'text' in c.lower()), None)
        label_col = next((c for c in raw.columns if 'disease' in c.lower() or 'diagnosis' in c.lower() or 'label' in c.lower() or 'condition' in c.lower()), None)

        if text_col and label_col:
            raw = raw[[text_col, label_col]].dropna()
            raw['label'] = raw[label_col].apply(map_kaggle_label)
            raw = raw[raw['label'].notna()]
            raw = raw.rename(columns={text_col: 'symptom_text'})[['symptom_text', 'label']]
            df_kaggle = raw
            print(f'Kept {len(df_kaggle)} Kaggle rows matching our 6 classes')
            print(df_kaggle['label'].value_counts())
        else:
            print(f'Could not identify text/label columns. text_col={text_col}, label_col={label_col}')
else:
    print('Skipping Kaggle augmentation (USE_KAGGLE=False). Using synthetic data only.')

# Combine
df = pd.concat([df_synthetic, df_kaggle], ignore_index=True)
df = df.dropna(subset=['symptom_text', 'label'])
df = df[df['label'].isin(CLASS_NAMES)]
print(f'\nTotal dataset: {len(df)} entries')
print(df['label'].value_counts())

## Section 3 — Preprocessing & Train/Test Split

In [ ]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['symptom_text'] = df['symptom_text'].apply(clean_text)
df = df.drop_duplicates(subset=['symptom_text'])

# Warn if any class is critically underrepresented
counts = df['label'].value_counts()
for cls in CLASS_NAMES:
    n = counts.get(cls, 0)
    if n < 40:
        print(f'WARNING: {cls} has only {n} samples — consider adding more descriptions')

le = LabelEncoder()
le.fit(CLASS_NAMES)
joblib.dump(le, LE_PATH)
print(f'Label encoder saved to {LE_PATH}')
print('Class order:', list(le.classes_))

texts  = df['symptom_text'].tolist()
labels = le.transform(df['label'].tolist())

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Train: {len(X_train_txt)} | Test: {len(X_test_txt)}')

## Section 4 — SBERT Encoding

In [ ]:
print('Loading SBERT model all-MiniLM-L6-v2 ...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding training texts ...')
X_train = sbert.encode(X_train_txt, batch_size=32, show_progress_bar=True)

print('Encoding test texts ...')
X_test  = sbert.encode(X_test_txt,  batch_size=32, show_progress_bar=True)

print(f'Train embeddings: {X_train.shape}  |  Test embeddings: {X_test.shape}')

## Section 5 — XGBoost Training

In [ ]:
# Use sample_weight to handle class imbalance (Hypersensitivity is underrepresented)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    reg_alpha=0.1,            # L1 regularisation
    reg_lambda=1.0,           # L2 regularisation
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(CLASS_NAMES),
    eval_metric='mlogloss',
    early_stopping_rounds=25,
    random_state=42,
    n_jobs=-1
)

print('Training XGBoost ...')
xgb_clf.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_pred = xgb_clf.predict(X_test)
print(f'\nTest accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

joblib.dump(xgb_clf, XGB_MODEL_PATH)
print(f'Model saved to {XGB_MODEL_PATH}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.title('XGBoost Text Model — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('./xgb_confusion_matrix.png', dpi=150)
plt.show()

## Section 6 — Multi-Variant Evaluation

5 held-out descriptions per class (never seen during training). Tests whether the model generalises to different phrasings — not just memorised training phrases.

In [ ]:
# Held-out test descriptions — NOT in symptom_data.csv
MULTI_VARIANT = {
    'Dermatitis': [
        'my dog has been scratching its belly raw for several days the skin looks red and inflamed',
        'there are weeping irritated patches on my dogs inner thighs it wont stop licking them',
        'my dogs ear flaps are red and hot to the touch it shakes its head frequently',
        'inflamed skin is developing along my dogs spine it scratches by rubbing against furniture',
        'my dog has crusty inflamed patches near its tail the skin is darkening from constant licking',
    ],
    'Fungal_infections': [
        'my dog has a persistent yeasty smell and the skin in its ear is dark and greasy',
        'the dogs paws smell strongly between the toes the skin is brown and very moist',
        'there is thick waxy brown discharge in my dogs ear it keeps scratching at it',
        'my dog has greasy flaky patches on its back the coat looks dull and smells musty',
        'the skin under my dogs belly has turned brownish and greasy a musty smell is present',
    ],
    'Healthy': [
        'my dog is in perfect health it has a shiny coat good appetite and no skin issues',
        'no symptoms at all the dog plays eats well and has a clean healthy coat',
        'my dog is active and happy with a full glossy coat and no visible skin problems',
        'the dog has no symptoms it eats drinks and plays normally with a healthy coat',
        'my dog shows no signs of illness the coat is thick and shiny with no lesions anywhere',
    ],
    'Hypersensitivity': [
        'my dog broke out in hives all over after eating something new its face is slightly swollen',
        'the dog developed a sudden widespread rash with raised welts after going outside',
        'my dogs face puffed up very quickly after a bee sting there are welts along its body',
        'there are sudden raised bumps appearing on my dogs skin it started an hour ago',
        'my dog has hives and swollen eyelids this started suddenly and is getting worse quickly',
    ],
    'demodicosis': [
        'my dog has bald patches forming around its muzzle the skin there looks scaly',
        'there is significant hair loss on my dogs forelegs and the skin looks rough and crusty',
        'my dogs face is losing fur in patches the exposed skin looks red and inflamed',
        'bald crusty patches are spreading from my dogs face to its neck and chest',
        'my dog has patchy fur loss with thickened rough skin the muzzle area is most affected',
    ],
    'ringworm': [
        'my dog has a perfectly round bald patch on its back with a crusty raised edge',
        'there are circular scaly lesions on my dog each one has a bald center and raised borders',
        'my dog developed a ring shaped bald patch on its neck with a crusty edge',
        'the dog has several round hairless spots with raised borders that are slowly spreading',
        'my dog has ring like lesions on its back the edges are raised and the patches look grey',
    ],
}

print('Multi-Variant Evaluation (5 descriptions per class, held out from training)\n')
print(f'{"Class":<22} {"Correct":<10} {"Total":<8} {"Per-Class Acc"}')
print('-' * 58)

all_correct = 0
all_total   = 0

for class_name, variants in MULTI_VARIANT.items():
    embeddings = sbert.encode(variants)
    preds      = xgb_clf.predict(embeddings)
    pred_labels = le.inverse_transform(preds)
    correct    = sum(p == class_name for p in pred_labels)
    total      = len(variants)
    all_correct += correct
    all_total   += total
    print(f'{class_name:<22} {correct:<10} {total:<8} {correct/total:.0%}')

print('-' * 58)
print(f'{"OVERALL":<22} {all_correct:<10} {all_total:<8} {all_correct/all_total:.0%}')

## Section 7 — Late Fusion with ResNet50

Combine ResNet50 image probabilities + XGBoost text probabilities via weighted averaging.
Grid search over 9 image/text weight combinations to find the optimal ratio.

In [ ]:
import tensorflow as tf
from pathlib import Path

if not Path(IMAGE_MODEL_PATH).exists():
    print(f'ERROR: ResNet50 model not found at {IMAGE_MODEL_PATH}')
    print('Place best_model.keras in the capstone root directory (parent of src/).')
    raise FileNotFoundError(IMAGE_MODEL_PATH)

print('Loading ResNet50 image model ...')
image_model = tf.keras.models.load_model(IMAGE_MODEL_PATH)
print('Image model loaded.')
image_model.summary()

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load test images — directory structure: IMAGE_DIR/test/{class_name}/image.jpg
test_image_dir = os.path.join(IMAGE_DIR, 'test')

if not os.path.isdir(test_image_dir):
    print(f'ERROR: Test image directory not found: {test_image_dir}')
    print('Update IMAGE_DIR in the config cell.')
    raise FileNotFoundError(test_image_dir)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_image_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='int'
)

# class_names from the dataset may differ in order from our CLASS_NAMES
ds_class_names = test_ds.class_names
print('Dataset class order:', ds_class_names)
print('Our class order:', list(le.classes_))

# Build index mapping: dataset index → our LabelEncoder index
ds_to_le = {i: le.transform([cls])[0] for i, cls in enumerate(ds_class_names) if cls in le.classes_}
print('Dataset→LE index mapping:', ds_to_le)

In [ ]:
# Get image probability vectors from ResNet50
print('Running ResNet50 inference on test images ...')

# Normalise pixel values (ResNet50 was trained with [0,1] normalisation)
normalization = tf.keras.layers.Rescaling(1.0 / 255)

all_img_probs = []   # shape (N, 6) — in dataset class order
all_true_labels = [] # true labels in dataset class order

for batch_images, batch_labels in test_ds:
    batch_norm = normalization(batch_images)
    probs = image_model.predict(batch_norm, verbose=0)  # (batch, 6)
    all_img_probs.append(probs)
    all_true_labels.extend(batch_labels.numpy())

all_img_probs   = np.vstack(all_img_probs)
all_true_labels = np.array(all_true_labels)

# Remap true labels from dataset order to LE order
all_true_le = np.array([ds_to_le[l] for l in all_true_labels])

# Remap image prob columns: dataset order → LE order
n_classes = len(CLASS_NAMES)
img_probs_reordered = np.zeros_like(all_img_probs)
for ds_idx, le_idx in ds_to_le.items():
    img_probs_reordered[:, le_idx] = all_img_probs[:, ds_idx]

img_acc = accuracy_score(all_true_le, np.argmax(img_probs_reordered, axis=1))
print(f'Image-only accuracy on test set: {img_acc:.4f}')
print(f'Total test images: {len(all_true_le)}')

In [ ]:
# Pair each test image with a held-out symptom description for its class
# Sample with replacement from the 5 multi-variant descriptions per class
paired_texts = []
for le_idx in all_true_le:
    class_name = le.inverse_transform([le_idx])[0]
    variants   = MULTI_VARIANT[class_name]
    paired_texts.append(random.choice(variants))

print(f'Encoding {len(paired_texts)} paired symptom texts ...')
txt_embeddings = sbert.encode(paired_texts, batch_size=32, show_progress_bar=True)

# XGBoost returns class probs in LE order
txt_probs = xgb_clf.predict_proba(txt_embeddings)  # (N, 6)

txt_acc = accuracy_score(all_true_le, np.argmax(txt_probs, axis=1))
print(f'Text-only accuracy on paired test set: {txt_acc:.4f}')

In [ ]:
# Late Fusion Grid Search
print('Late Fusion Grid Search\n')
print(f'{"img_weight":<14} {"txt_weight":<14} {"Accuracy"}')
print('-' * 44)

best_acc = 0.0
best_weights = (0.5, 0.5)
grid_results = []

for img_w_pct in range(1, 10):  # 0.1 to 0.9
    img_w = img_w_pct / 10
    txt_w = 1.0 - img_w
    fused_probs = img_w * img_probs_reordered + txt_w * txt_probs
    preds       = np.argmax(fused_probs, axis=1)
    acc         = accuracy_score(all_true_le, preds)
    grid_results.append((img_w, txt_w, acc))
    print(f'{img_w:<14.1f} {txt_w:<14.1f} {acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        best_weights = (img_w, txt_w)

print('-' * 44)
print(f'\nBest weights — image: {best_weights[0]:.1f}, text: {best_weights[1]:.1f}')
print(f'Best fused accuracy: {best_acc:.4f}')

In [ ]:
# Full report at best fusion weights
img_w_best, txt_w_best = best_weights
fused_best = img_w_best * img_probs_reordered + txt_w_best * txt_probs
preds_best = np.argmax(fused_best, axis=1)

print(f'=== Final Late Fusion Report (image_w={img_w_best}, text_w={txt_w_best}) ===')
print(classification_report(all_true_le, preds_best, target_names=le.classes_))

In [ ]:
# Fusion accuracy curve
img_weights = [r[0] for r in grid_results]
accs        = [r[2] for r in grid_results]

plt.figure(figsize=(8, 5))
plt.plot(img_weights, accs, marker='o', linewidth=2, color='steelblue')
plt.axhline(img_acc, linestyle='--', color='orange', label=f'Image only ({img_acc:.3f})')
plt.axhline(txt_acc, linestyle='--', color='green',  label=f'Text only  ({txt_acc:.3f})')
plt.axvline(best_weights[0], linestyle=':', color='red', label=f'Best img_w={best_weights[0]}')
plt.xlabel('Image weight (text weight = 1 - image weight)')
plt.ylabel('Accuracy')
plt.title('Late Fusion Grid Search — Accuracy vs Image Weight')
plt.legend()
plt.tight_layout()
plt.savefig('./late_fusion_grid_search.png', dpi=150)
plt.show()

print('\nSummary')
print(f'  ResNet50 (image only)  : {img_acc:.4f}')
print(f'  XGBoost  (text only)   : {txt_acc:.4f}')
print(f'  Late fusion (best)     : {best_acc:.4f}  [img_w={best_weights[0]}, txt_w={best_weights[1]}]')